In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

## 입력

In [ ]:
DATASETS = [
    {
        "name": "A_5m",
        "csv_path": Path(
            r"D:\uwb-config\calibration\data\A_5m.csv"
        ),
        "target_distance_m": 5.0,
    },
    {
        "name": "B_10m",
        "csv_path": Path(
            r"D:\uwb-config\calibration\data\B_10m.csv"
        ),
        "target_distance_m": 10.0,
    },
    {
        "name": "C_20m",
        "csv_path": Path(
            r"D:\uwb-config\calibration\data\C_20m.csv"
        ),
        "target_distance_m": 20.0,
    },
]

current_ant_delay = 16450
trim_seconds = 5
anchor_id = 6

# 4: tag TX/RX + anchor TX/RX 모두 같은 tick만큼 변경
# 2: tag 또는 anchor 한쪽 보드의 TX/RX만 변경
adjusted_delay_registers = 2

# 거리별 데이터를 같은 비율로 반영할지 선택
# "equal"          : 5 m, 10 m, 20 m 오차를 각각 동일 가중치로 평균
# "packet_weighted": 전체 패킷 수를 기준으로 가중 평균
bias_aggregation = "equal"

In [ ]:
# =========================================================
# DW3000 시간·거리 단위
# =========================================================

DWT_TIME_UNITS = 1.0 / (499.2e6 * 128.0)
SPEED_OF_LIGHT = 299_702_547.0
M_PER_DTU = DWT_TIME_UNITS * SPEED_OF_LIGHT

# 안테나 지연 레지스터를 1 tick 변경했을 때
# 측정 거리에 나타나는 예상 변화량
range_per_tick = -(adjusted_delay_registers / 2.0) * M_PER_DTU

## CSV 파일 분석

In [ ]:

def analyze_calibration_file(
    name,
    csv_path,
    target_distance_m,
    anchor_id=None,
    trim_seconds=0,
):
    df = pd.read_csv(csv_path)
    total_packets = len(df)

    required_columns = {"Timestamp", "Dist"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{name}: 필요한 열이 없습니다: {sorted(missing_columns)}"
        )

    # 특정 앵커 데이터만 선택
    if anchor_id is not None:
        if "ANC" not in df.columns:
            raise ValueError(f"{name}: ANC 열이 없습니다.")

        df = df[df["ANC"] == anchor_id].copy()

        if df.empty:
            raise ValueError(
                f"{name}: ANC == {anchor_id}인 데이터가 없습니다."
            )

    filtered_packets = len(df)

    # 거리값 숫자 변환
    df["Dist"] = pd.to_numeric(df["Dist"], errors="coerce")

    # 시간 변환
    df["Timestamp_dt"] = pd.to_datetime(
        df["Timestamp"],
        format="%H:%M:%S.%f",
        errors="coerce",
    )

    df = df.dropna(subset=["Timestamp_dt", "Dist"]).copy()

    if df.empty:
        raise ValueError(
            f"{name}: 유효한 Timestamp 또는 Dist 데이터가 없습니다."
        )

    # CSV의 원래 순서 유지
    df = df.reset_index(drop=True)

    # 자정을 넘어가는 데이터 보정
    time_diff = df["Timestamp_dt"].diff().dt.total_seconds()
    day_offset = (time_diff < 0).cumsum()

    df["Timestamp_dt"] = (
        df["Timestamp_dt"]
        + pd.to_timedelta(day_offset, unit="D")
    )

    start_t = df["Timestamp_dt"].iloc[0]
    end_t = df["Timestamp_dt"].iloc[-1]

    trim_start = start_t + pd.Timedelta(seconds=trim_seconds)
    trim_end = end_t - pd.Timedelta(seconds=trim_seconds)

    if trim_start > trim_end:
        raise ValueError(
            f"{name}: 데이터 길이가 짧아 앞뒤 "
            f"{trim_seconds}초를 제거할 수 없습니다."
        )

    mid = df[
        (df["Timestamp_dt"] >= trim_start)
        & (df["Timestamp_dt"] <= trim_end)
    ].copy()

    if mid.empty:
        raise ValueError(
            f"{name}: trim 이후 분석할 데이터가 없습니다."
        )

    mean_dist = mid["Dist"].mean()
    std_dist = mid["Dist"].std()

    # 일반적인 거리 bias 정의
    # 양수: 실제보다 길게 측정
    # 음수: 실제보다 짧게 측정
    bias_m = mean_dist - target_distance_m

    # 측정 거리에 적용되어야 할 보정량
    correction_m = target_distance_m - mean_dist

    # 이 데이터 하나만 사용했을 때 필요한 tick
    individual_delta_tick = correction_m / range_per_tick

    result = {
        "name": name,
        "csv_path": str(csv_path),
        "target_m": target_distance_m,
        "mean_dist_m": mean_dist,
        "std_m": std_dist,
        "bias_m": bias_m,
        "correction_m": correction_m,
        "individual_delta_tick": individual_delta_tick,
        "total_packets": total_packets,
        "filtered_packets": filtered_packets,
        "used_packets": len(mid),
        "start_timestamp": mid["Timestamp"].iloc[0],
        "end_timestamp": mid["Timestamp"].iloc[-1],
    }

    if "#Packet" in mid.columns:
        result["start_packet"] = mid["#Packet"].iloc[0]
        result["end_packet"] = mid["#Packet"].iloc[-1]
    else:
        result["start_packet"] = np.nan
        result["end_packet"] = np.nan

    return result

## 거리 데이터 분석

In [ ]:
rows = []

for dataset in DATASETS:
    result = analyze_calibration_file(
        name=dataset["name"],
        csv_path=dataset["csv_path"],
        target_distance_m=dataset["target_distance_m"],
        anchor_id=anchor_id,
        trim_seconds=trim_seconds,
    )

    rows.append(result)

summary = pd.DataFrame(rows)


# =========================================================
# 여러 거리의 bias 통합
# =========================================================

# 각 거리 데이터를 동일한 비중으로 반영
equal_mean_bias_m = summary["bias_m"].mean()
equal_mean_correction_m = summary["correction_m"].mean()

# 패킷 개수 기준 가중 평균
packet_weighted_bias_m = np.average(
    summary["bias_m"],
    weights=summary["used_packets"],
)

packet_weighted_correction_m = np.average(
    summary["correction_m"],
    weights=summary["used_packets"],
)

if bias_aggregation == "equal":
    final_bias_m = equal_mean_bias_m
    final_correction_m = equal_mean_correction_m

elif bias_aggregation == "packet_weighted":
    final_bias_m = packet_weighted_bias_m
    final_correction_m = packet_weighted_correction_m

else:
    raise ValueError(
        "bias_aggregation은 'equal' 또는 "
        "'packet_weighted'여야 합니다."
    )


# =========================================================
# 최종 안테나 지연값 계산
# =========================================================

delta_tick = final_correction_m / range_per_tick
suggested_ant_delay = int(
    round(current_ant_delay + delta_tick)
)

# 공통 보정을 적용했을 때 예상 거리
summary["predicted_dist_after_m"] = (
    summary["mean_dist_m"]
    + delta_tick * range_per_tick
)

summary["predicted_bias_after_m"] = (
    summary["predicted_dist_after_m"]
    - summary["target_m"]
)


## 출력

In [ ]:
print("\n[거리별 결과]")
print(
    summary[
        [
            "name",
            "target_m",
            "mean_dist_m",
            "bias_m",
            "individual_delta_tick",
            "used_packets",
        ]
    ].round(4).to_string(index=False)
)

print("\n[최종 보정 결과]")
print(f"사용 방식       : {bias_aggregation}")
print(f"평균 bias       : {final_bias_m:+.4f} m")
print(f"거리 보정량     : {final_correction_m:+.4f} m")
print(f"추천 보정량     : {delta_tick:+.2f} tick")
print(f"현재 delay      : {current_ant_delay}")
print(f"추천 delay      : {suggested_ant_delay}")

print("\n[보정 후 예상 결과]")
print(
    summary[
        [
            "name",
            "target_m",
            "predicted_dist_after_m",
            "predicted_bias_after_m",
        ]
    ].round(4).to_string(index=False)
)